In [ ]:
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
import matplotlib as mpl
import matplotlib.patheffects as mpe
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import rasterio as rio
import seaborn as sns
from tqdm import tqdm
import xarray as xr
from scipy.stats import mannwhitneyu, ttest_ind, wilcoxon

from project.utils.data import enrich, load_hillshade, sel_subset
from project.utils.metrics import compute_metrics
from project.utils.plotting import ProjectCustomExtent, despine, cbar_halo

plt.rcParams["figure.dpi"] = 150

# Switzerland LV95 with extended limits to cover 'Hydrological Switzerland'.
lv95 = ProjectCustomExtent(2056, [2474273, 2842000, 1058000, 1325000])

In [ ]:
# Load predictions.
ds = xr.open_zarr('/net/argon/landclim/kraftb/machflow_routing/data/results/predictions.zarr').sel(
    time=slice('1993', None))

# Load HRU polygons.
hrus = gpd.read_file(
    '/net/argon/landclim/kraftb/machflow_routing/data/processed/hrus/hrus.gpkg', use_arrow=True).set_index('EZGNR')

ds['R_pred_mean'] = ds['R_pred'].mean('time').compute()

ds_pretrain = xr.open_dataset('/net/argon/landclim/kraftb/machflow_routing/logs/pretraining/xval/fold_000/preds.zarr')

rivers = gpd.read_file('/net/argon/landclim/kraftb/machflow_routing/data/raw/rivernet', use_arrow=True)

rivers_subset = rivers.loc[rivers.CUM_LEN > 20000]

bg = load_hillshade(
    hillshade_dir='/net/argon/landclim/kraftb/machflow_routing/data/processed/',
    hrus=hrus,
    ds=ds,
    use_terrain=False,
    force_recompute=False)

In [ ]:
hrus_lakes = hrus.loc[ds.hru.where(ds.is_lake.load() == 1, drop=True)]

In [ ]:
not_lake = ds.is_lake.load() == 0
ds = ds.where(not_lake, drop=True)
ds_pretrain = ds_pretrain.where(not_lake, drop=True)

In [ ]:
corr_rr_prevah = xr.corr(ds.q_prevah_mmd, ds.q_pred_mmd.sel(model='routed_random'), dim='time').compute()
corr_rr_pretraining = xr.corr(ds_pretrain.q_pred_mmd, ds.q_pred_mmd.sel(model='routed_random'), dim='time').compute()

bias_rr_prevah = (
    ds.q_pred_mmd.sel(model='routed_random').mean('time') - ds.q_prevah_mmd.mean('time')).compute()
bias_rr_pretraining = (
    ds.q_pred_mmd.sel(model='routed_random').mean('time') - ds_pretrain.q_pred_mmd.mean('time')).compute()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3), subplot_kw={'projection': lv95}, gridspec_kw={'wspace': 0.01})

bodr = cartopy.feature.NaturalEarthFeature(category='cultural', 
    name='admin_0_boundary_lines_land', scale='10m', facecolor='none')

for i, (rr, title) in enumerate(
        zip(
            [corr_rr_pretraining, bias_rr_pretraining],
            ['Correlation Pre-trained vs. Post-trained', 'Bias Pre-trained vs. Post-trained'])
    ):

    ax = axes[i]
    ax.set_global()

    if i ==0:
        norm = mpl.colors.Normalize(vmin=0.85, vmax=1.0)
        cmap = sns.color_palette('rocket', as_cmap=True)
        label = 'Correlation (-)'
        extend = 'min'
    else:
        norm = mpl.colors.Normalize(vmin=-1, vmax=1)
        # cmap = sns.color_palette("vlag", as_cmap=True)
        cmap = sns.color_palette('PiYG', as_cmap=True)
        label = 'Bias (mm d$^{-1}$)' 
        extend = 'both'

    axins = ax.inset_axes((0.04, 0.88, 0.25, 0.03))
    im = enrich(
        ds=rr,
        hrus=hrus,
        variables='val'
    ).plot(
        column='val',
        legend=True,
        ec='k',
        lw=0.1,
        alpha=0.6,
        ax=ax,
        norm=norm,
        cmap=cmap,
        legend_kwds={
            'cax': axins, 'orientation': 'horizontal', 'label': label,
            'extend': extend}
    )

    cbar_halo(axins, which='x', linewidth=5, color='0.95')

    rivers_subset.plot(
        ec='k', fc='none', linewidth=(np.log(rivers_subset.CUM_LEN + 1) / 12 - 0.5),
        alpha=0.8,
        zorder=5,
        ax=ax)

    hrus_lakes.plot(ec='none', fc='lightblue', zorder=6, ax=ax)

    ax.set_title(title, fontsize=8)

fig.savefig('./figures/preva_comparison.png', dpi=300, bbox_inches='tight')

## Runoff robustness

In [ ]:
dr = (ds.q_pred_mmd.sel(model='routed_random') - ds_pretrain.q_pred_mmd).compute()
de = (ds.q_pred_mmd.sel(model='routed_easy') - ds_pretrain.q_pred_mmd).compute()
dh = (ds.q_pred_mmd.sel(model='routed_hard') - ds_pretrain.q_pred_mmd).compute()

In [ ]:
cre = xr.corr(dr, de, dim='time').compute()
crh = xr.corr(dr, dh, dim='time').compute()
ceh = xr.corr(de, dh, dim='time').compute()

In [ ]:
cbre = xr.corr(dr.mean(dim='time'), de.mean(dim='time')).compute()
cbrh = xr.corr(dr.mean(dim='time'), dh.mean(dim='time')).compute()
cbeh = xr.corr(de.mean(dim='time'), dh.mean(dim='time')).compute()

In [ ]:
print('Temporal correlation of runoff delta to PREVAH')

for (c, n) in zip([cre, crh, ceh], ['cre', 'crh', 'ceh']):
    print(f'{n}: {c.mean().item():0.2f} $\pm$ {c.std().item():0.2f}')

print('Spatial correlation of biases towards PREVAH')

for (b, n) in zip([cbre, cbrh, cbeh], ['bre', 'brh', 'beh']):
    print(f'{n}: {b.item():0.2f}')

## Retention robustness

In [ ]:
rcre = xr.corr(ds.R_pred.sel(model='routed_random'), ds.R_pred.sel(model='routed_easy'), dim='time').compute()
rcrh = xr.corr(ds.R_pred.sel(model='routed_random'), ds.R_pred.sel(model='routed_hard'), dim='time').compute()
rceh = xr.corr(ds.R_pred.sel(model='routed_easy'), ds.R_pred.sel(model='routed_hard'), dim='time').compute()

In [ ]:
rscre = xr.corr(ds.R_pred_mean.sel(model='routed_random'), ds.R_pred_mean.sel(model='routed_easy')).compute()
rscrh = xr.corr(ds.R_pred_mean.sel(model='routed_random'), ds.R_pred_mean.sel(model='routed_hard')).compute()
rsceh = xr.corr(ds.R_pred_mean.sel(model='routed_easy'), ds.R_pred_mean.sel(model='routed_hard')).compute()

In [ ]:
print('Temporal correlation of learned retention')

for (c, n) in zip([rcre, rcrh, rceh], ['rcre', 'rcrh', 'rceh']):
    print(f'{n}: {c.mean().item():0.2f} $\pm$ {c.std().item():0.2f}')

print('Spatial correlation of mean learned retention')

for (b, n) in zip([rscre, rscrh, rsceh], ['rscre', 'rscrh', 'rsceh']):
    print(f'{n}: {b.item():0.2f}')